In [ ]:
!pip install datasets pandas

In [ ]:
import os
import json
from datasets import load_dataset

# Ensure directories exist
os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/cleaned", exist_ok=True)

def format_to_sharegpt(system_prompt, user_text, assistant_text):
    """Formats data into the standard conversational schema Unsloth expects."""
    return {
        "conversations": [
            {"from": "system", "value": system_prompt},
            {"from": "user", "value": user_text},
            {"from": "assistant", "value": assistant_text}
        ]
    }

print("1. Processing Text Generation Data...")
# Using an open-source general instruction dataset (e.g., Alpaca or similar)
text_dataset = load_dataset("yahma/alpaca-cleaned", split="train[:5000]") # Limit to 5k for fast local training
text_cleaned = []

for row in text_dataset:
    # Only take rows with instructions and outputs
    prompt = f"{row['instruction']}\n{row.get('input', '')}".strip()
    formatted = format_to_sharegpt(
        "You are a helpful AI assistant. Provide clear, concise answers.",
        prompt,
        row['output']
    )
    text_cleaned.append(formatted)

with open("../data/cleaned/text_data.jsonl", "w") as f:
    for item in text_cleaned:
        f.write(json.dumps(item) + "\n")


print("2. Processing Reasoning/Math Data...")
# Using a chain-of-thought math dataset
reason_dataset = load_dataset("gsm8k", "main", split="train[:5000]")
reason_cleaned = []

for row in reason_dataset:
    formatted = format_to_sharegpt(
        "You are a logical reasoning AI. Think step-by-step before answering.",
        row['question'],
        row['answer']
    )
    reason_cleaned.append(formatted)

with open("../data/cleaned/reason_data.jsonl", "w") as f:
    for item in reason_cleaned:
        f.write(json.dumps(item) + "\n")

print("Data processing complete. Saved to data/cleaned/")